# Open Challenge 3 — Save the Qubit!
**Mission:** implement a three-qubit bit-flip repetition code with syndrome extraction and correction, then quantify its benefit under independent noise. **Estimated time:** 5–10 hours. **100 points.**

**Core physics:** `|0_L⟩=|000⟩`, `|1_L⟩=|111⟩`. A single X error is detected by parity checks `Z0Z1` and `Z1Z2`. For independent physical X errors with probability p, ideal logical failure is `3p²−2p³`. **Part 2** investigates dephasing and constructs a complementary phase-flip repetition code; the bit-flip code alone does **not** protect against phase errors.

**Deliverables:** runnable encoding, syndrome measurements, correction, tests for each single-qubit error, error-rate plots for bit flips and dephasing, README explaining assumptions and limitations. No IBM account, hardware, or cloud needed.

## Local setup
Run once with internet, then work fully offline. No IBM account or token. Python 3.10+ recommended.

```python
%pip install qiskit qiskit-aer numpy scipy matplotlib networkx
```
Restart kernel after installation. The following cells assume packages are installed.

In [ ]:
%pip install qiskit qiskit-aer numpy scipy matplotlib networkx

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator, state_fidelity
from qiskit_aer import AerSimulator
SEED = 42
SHOTS = 2048
print("Ready: local Qiskit Aer; seed", SEED)

## Syndrome table
Syndrome is shown in order `(S1,S2)` where S1=Z0Z1 and S2=Z1Z2; 0 means parity +1 and 1 means parity −1.

| Error | S1 | S2 | Syndrome |
|---|---:|---:|---|
| None | +1 | +1 | 00 |
| X0 | −1 | +1 | 10 |
| X1 | −1 | −1 | 11 |
| X2 | +1 | −1 | 01 |

Qiskit prints classical bitstrings in reverse register order, so explicitly decode the two syndrome bits.

In [ ]:
def encode_logical_zero():
    qc = QuantumCircuit(3)
    # |000> is already the encoded logical zero
    return qc

def encode_logical_plus():
    qc = QuantumCircuit(3)
    # TODO: prepare (|000>+|111>)/sqrt(2)
    return qc

print("Logical zero:", Statevector.from_instruction(encode_logical_zero()).probabilities_dict())
print("Your logical plus:", Statevector.from_instruction(encode_logical_plus()).probabilities_dict())

## Phase 1 — Build the syndrome circuit
Use 3 data qubits and 2 ancillas. Measure `Z0Z1` with CNOTs from data 0 and 1 to ancilla 3; measure `Z1Z2` with CNOTs from data 1 and 2 to ancilla 4. Measure ancillas into two classical bits. Do **not** measure the data qubits during syndrome extraction.

The supplied skeleton has ancilla qubits indexed 3 and 4. `measure(3,0)` stores S1 in classical bit 0 and `measure(4,1)` stores S2 in classical bit 1.

In [ ]:
def syndrome_circuit(error_qubit=None):
    qc = QuantumCircuit(5, 2)
    # Data q0,q1,q2; ancilla q3,q4
    # Encoded |0_L> is |000>
    if error_qubit is not None:
        qc.x(error_qubit)

    # TODO: parity check Z0Z1: CNOT(data0 -> ancilla3), CNOT(data1 -> ancilla3)
    # TODO: parity check Z1Z2: CNOT(data1 -> ancilla4), CNOT(data2 -> ancilla4)

    qc.measure(3, 0)  # S1
    qc.measure(4, 1)  # S2
    return qc

sim = AerSimulator(seed_simulator=SEED)
for error in [None, 0, 1, 2]:
    counts = sim.run(syndrome_circuit(error), shots=256).result().get_counts()
    print("Error", error, "Qiskit bitstring (S2 S1):", counts)

## Phase 2 — Decode syndrome and correct
Use a lookup from `(S1,S2)` to which data qubit should receive an X correction. You may implement **classical feed-forward** using Qiskit control flow, or a two-stage experiment that reads the syndrome and constructs a recovery circuit. For a truly arbitrary logical qubit, avoid measuring the data qubits to decide corrections.

**Validation:** run each single-qubit X fault and verify that correction returns the logical data to the encoded state.

In [ ]:
def decode_syndrome(s1, s2):
    # TODO: return None for 00, or 0/1/2 for other syndromes
    raise NotImplementedError

def correct_data_circuit(s1, s2):
    qc = QuantumCircuit(3)
    # TODO: call decode_syndrome and apply X to appropriate qubit
    raise NotImplementedError

## Phase 3 — Independent noise experiment
For **ideal** syndrome extraction/correction, logical failure occurs if 2 or 3 of the three data qubits flip. Compare Monte Carlo samples against the theoretical `3p²−2p³` curve and unencoded error `p`.

The following Monte Carlo is a **classical reference model** of independent Pauli-X fault events; it is not a substitute for the required Qiskit syndrome and recovery circuit. Use it to sanity-check your implementation.

In [ ]:
def theoretical_logical_error(p):
    return 3*p*p - 2*p**3

rng = np.random.default_rng(SEED)
ps = np.linspace(0, 0.5, 11)
trials = 10000
reference = []
for p in ps:
    faults = rng.random((trials, 3)) < p
    logical_failure = (faults.sum(axis=1) >= 2)
    reference.append(logical_failure.mean())

plt.figure(figsize=(8,4))
plt.plot(ps, ps, "--", label="Unencoded error p")
plt.plot(ps, theoretical_logical_error(ps), label="Ideal logical theory")
plt.scatter(ps, reference, label="Monte Carlo fault-pattern reference")
plt.xlabel("Physical bit-flip probability p")
plt.ylabel("Failure probability")
plt.title("Three-qubit bit-flip code: idealized reference")
plt.grid(alpha=.3); plt.legend(); plt.show()

## Phase 4 — Full Qiskit experiment (your work)
Implement an end-to-end Qiskit simulation with explicit Pauli-X faults, syndrome extraction, correction, and final logical verification. Choose several p values including 0, 0.05, 0.1, 0.2 and 0.3. Use enough shots to estimate error rates with useful precision. Compare against theory.

For arbitrary logical states, verify coherence with `|+_L⟩` as well as `|0_L⟩`. For optional noisy gates and measurements, state your additional noise assumptions: the ideal formula no longer applies exactly.

In [ ]:
def run_full_qec_experiment(p, shots=2048, seed=42):
    # TODO: simulate independent X errors on the three data qubits;
    # extract syndrome using ancillas; correct; verify logical result.
    # Return a logical failure estimate in [0,1].
    raise NotImplementedError

# TODO: after implementation, uncomment
# measured = [run_full_qec_experiment(float(p)) for p in ps]
# plt.plot(ps, measured, "o-", label="Qiskit QEC experiment")

---
# Part 2 — Dephasing noise and protecting quantum phase

**Why this matters:** Bit flips (`X`) exchange `|0⟩` and `|1⟩`, whereas a phase flip (`Z`) changes the *relative phase* of a superposition. Pure dephasing suppresses coherence without necessarily changing computational-basis populations. Therefore, testing only `|0⟩` cannot reveal dephasing: use `|+⟩=(|0⟩+|1⟩)/√2` and measure in the **X basis** (apply H before measuring Z).

**Part 2 mission:** (1) visualize dephasing on an unencoded `|+⟩`; (2) implement the 3-qubit **phase-flip repetition code**, with logical basis `|0_L⟩=|+++⟩`, `|1_L⟩=|---⟩`; (3) extract `X0X1` and `X1X2` syndromes; (4) correct any one `Z` error; (5) compare logical and unencoded error rates for independent Z faults. As a stretch goal, study continuous phase damping and noisy syndrome extraction.

**Noise models are not identical:** A stochastic Pauli-Z error with probability `p` is a *phase-flip channel*. A phase-damping channel with parameter `γ` is a *dephasing channel*. Both reduce off-diagonal coherence, but their parameters and maps differ. In the code below, we explore **both**, without equating `p` and `γ`.

All experiments use local Qiskit Aer. No IBM account, token, or hardware is needed.


## Part 2A — See dephasing on the Bloch sphere / in the X basis

A pure `|+⟩` state has density matrix `ρ = [[1/2,1/2],[1/2,1/2]]`. Under ideal phase damping of strength `γ`, the off-diagonal terms are multiplied by `sqrt(1-γ)` while the diagonal populations stay `1/2`. The expected X-basis `+` probability is `(1 + sqrt(1-γ))/2`.

Use Aer density-matrix simulation to verify this. The `id` gate is an explicit noise location; run it **without transpilation** so it is not optimized away.


In [ ]:
from qiskit_aer.noise import NoiseModel, phase_damping_error, pauli_error
from qiskit.quantum_info import DensityMatrix


def phase_damping_experiment(gamma, shots=4096, seed=SEED):
    """Prepare |+>, apply phase damping after id, measure in X basis."""
    noise = NoiseModel()
    noise.add_all_qubit_quantum_error(phase_damping_error(gamma), ["id"])
    circuit = QuantumCircuit(1, 1)
    circuit.h(0)
    circuit.id(0)  # Explicit noise location: do not transpile this away
    circuit.h(0)   # X-basis measurement
    circuit.measure(0, 0)
    simulator = AerSimulator(method="density_matrix", noise_model=noise, seed_simulator=seed)
    counts = simulator.run(circuit, shots=shots).result().get_counts()
    return counts.get("0", 0) / shots


gammas = np.linspace(0, 1, 11)
observed_plus = [phase_damping_experiment(float(g)) for g in gammas]
theory_plus = (1 + np.sqrt(1 - gammas)) / 2

plt.figure(figsize=(8, 4))
plt.plot(gammas, theory_plus, label="Phase damping theory: P(+)")
plt.scatter(gammas, observed_plus, label="Aer sampled P(+)")
plt.axhline(0.5, color="gray", linestyle="--", label="Fully dephased")
plt.xlabel("Phase damping parameter γ")
plt.ylabel("Probability of + in X basis")
plt.title("Dephasing destroys coherence, not Z-basis populations")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## Part 2B — Build the three-qubit phase-flip code

The phase-flip code is the bit-flip code expressed in the Hadamard-rotated basis. To encode an arbitrary input `α|0⟩+β|1⟩`, first perform the bit-flip encoding (`cx(0,1)` and `cx(0,2)`), then apply H to **all three data qubits**. A single `Z` fault becomes an `X` fault in this rotated basis.

The stabilizers are `S1=X0X1`, `S2=X1X2`. Their syndrome table for a single Z error is the same as the X-error table in Part 1:

| Fault | S1 | S2 | syndrome `(s1,s2)` | recovery |
|---|---:|---:|---|---|
| None | +1 | +1 | `00` | None |
| `Z(0)` | −1 | +1 | `10` | `Z(0)` |
| `Z(1)` | −1 | −1 | `11` | `Z(1)` |
| `Z(2)` | +1 | −1 | `01` | `Z(2)` |

**Important:** For the `X0X1` parity measurement, put H on each data qubit involved *before and after* the two CNOTs into the ancilla (or use an equivalent ancilla-in-X-basis construction). Use fresh ancillas for both checks. Qiskit prints two classical bits in order `s2 s1` when `s1` is stored in classical bit 0.


In [ ]:
def encode_phase_logical_plus():
    """TODO: encode logical |+_L> = (|+++> + |--->)/sqrt(2)."""
    qc = QuantumCircuit(3)
    # TODO: prepare |+> on qubit 0, then encode using CX and H gates
    return qc


def phase_syndrome_circuit(error_qubit=None):
    """TODO: measure X0X1 and X1X2 using ancillas 3 and 4."""
    qc = QuantumCircuit(5, 2)
    # TODO: prepare a valid encoded logical state on data q0,q1,q2
    # TODO: if error_qubit is not None, apply Z to that data qubit
    # TODO: extract S1=X0X1 to ancilla q3, S2=X1X2 to ancilla q4
    qc.measure(3, 0)  # S1
    qc.measure(4, 1)  # S2
    return qc


def decode_phase_syndrome(s1, s2):
    """TODO: return None or data-qubit index 0, 1, 2."""
    raise NotImplementedError


def phase_recovery(s1, s2):
    """TODO: return a 3-qubit circuit with the inferred Z correction."""
    raise NotImplementedError

# TODO: Run each of the four cases and verify syndromes: None, 0, 1, 2
# for error in [None, 0, 1, 2]:
#     counts = sim.run(phase_syndrome_circuit(error), shots=256).result().get_counts()
#     print(error, counts)


## Part 2C — Quantify protection from independent phase flips

For **independent stochastic Z faults** with probability `p` on each data qubit and **perfect** encoding/syndrome extraction/recovery, the logical phase-flip failure rate is the probability of two or three physical Z faults:

`P_L(p) = 3p²(1-p) + p³ = 3p² - 2p³`.

The unencoded `|+⟩` phase-flip failure rate is `p`. This formula is **not** the generic phase-damping formula. For the extension to phase damping, specify the channel precisely and measure logical-state fidelity or logical X expectation.

**Student tasks:** create a full Qiskit Aer experiment with injected Z faults; measure the two stabilizers; apply recovery; decode and measure in a suitable basis. Plot unencoded and corrected logical failure against `p`, including statistical error bars if possible.


In [ ]:
def run_phase_flip_qec_experiment(p, shots=2048, seed=SEED):
    """TODO: estimate logical failure under independent Pauli-Z errors."""
    # Suggestions:
    # 1. Prepare encoded logical |+_L> or another state sensitive to phase errors.
    # 2. Insert independent Z errors on each of 3 data qubits.
    # 3. Extract X-parity syndromes with ancillas and recover with Z.
    # 4. Decode and measure an appropriate logical observable.
    # 5. Return a failure fraction between 0 and 1.
    raise NotImplementedError

ps_phase = np.linspace(0, 0.5, 11)
plt.figure(figsize=(8, 4))
plt.plot(ps_phase, ps_phase, "--", label="Unencoded Z-error rate")
plt.plot(ps_phase, 3 * ps_phase**2 - 2 * ps_phase**3,
         label="Ideal phase-flip code logical failure")
# TODO: Uncomment after implementing the experiment:
# measured = [run_phase_flip_qec_experiment(float(p)) for p in ps_phase]
# plt.scatter(ps_phase, measured, label="Aer phase-flip QEC")
plt.xlabel("Physical phase-flip probability p")
plt.ylabel("Failure probability")
plt.title("Phase-flip error correction: theoretical target")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## Part 2D — Optional research extensions

1. **Continuous dephasing:** Use `phase_damping_error(γ)` at explicit noise locations. Compare coherence/fidelity of encoded and unencoded logical states; do **not** assume `γ=p`.
2. **Imperfect checks:** Add noise to syndrome extraction and readout, then determine when the added correction overhead makes the code worse.
3. **Repeated rounds:** Explore syndrome-history decoding rather than trusting a single noisy measurement.
4. **Code limitations:** Show that the bit-flip code from Part 1 does not correct a general Z fault, and the phase-flip code does not correct a general X fault.

**Self-check:** Does your state preserve normalization? Are syndromes consistent with qubit ordering? Does the logical error curve follow `3p²−2p³` under the stated ideal model? Can you explain why Z-basis measurements alone cannot detect dephasing on `|+⟩`?


## Self-test checklist
- Syndrome table is correct for no error and each single X error.
- Recovery corrects any **single** X error.
- `|+_L⟩` is coherent and normalized, not a classical mixture.
- Full experiment agrees with `3p²−2p³` within statistical error for ideal recovery.
- Explain why multiple faults can be misdiagnosed.
- Distinguish data noise from ancilla, gate, and readout noise.

**Stretch:** phase-flip repetition code, repeated rounds, noisy syndrome extraction, 9-qubit Shor code, logical fidelity for arbitrary input states.
- Part 2: phase damping reduces X-basis coherence while Z-basis populations remain unchanged.
- Part 2: X-parity syndromes locate single Z errors; phase recovery restores encoded state.
- Part 2: compare phase-flip logical error with the ideal theoretical curve.


## Judging (100 points total)
| Criterion | Points |
|---|---:|
| Part 1: bit-flip encoding, syndrome and correction | 30 |
| Part 1: noise comparison and interpretation | 15 |
| Part 2: phase-damping demonstration and interpretation | 15 |
| Part 2: phase-flip encoding, X stabilizers and recovery | 25 |
| Part 2: phase-noise benchmark and comparison | 10 |
| Reproducibility, README and presentation | 5 |

**Scope note:** The three-qubit bit-flip code corrects one X fault; the rotated phase-flip code corrects one Z fault. Neither corrects arbitrary single-qubit errors by itself. A physical phase-damping channel and a stochastic Pauli-Z channel must be identified separately.
